# T12: RAG as a Tool

The RAG agent in T01 always retrieves before answering. Here retrieval is a **tool** instead: the LLM decides when to call it and with which query. It answers general questions from its own knowledge and retrieves for questions about PrimePack.

This notebook runs the loop by hand, as in T10. T13 puts the same tool into a `ToolAgent`.

## Setup

Run T01 first: it builds the vector store at `backend/db/data_vs.db`. If the store is empty, `load_or_build_vector_store()` builds it from all documents, which takes several minutes.

In [ ]:
import json
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from ki_kompass_rag.baseline_rag import RETRIEVER_TOP_K, load_or_build_vector_store
from ki_kompass_rag.tool_agents import RetrieveRelevantChunks

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
retriever = VectorStoreRetriever(embedding_model, vector_store, top_k=RETRIEVER_TOP_K)


def text_message(role: Roles, text: str) -> LLMMessage:
    return LLMMessage(role=role, content=[MessageContent(type="text", text=text)])

## The retrieval tool

`RetrieveRelevantChunks` retrieves the top chunks for the `query` argument chosen by the LLM. It returns them as text under `result`, and the raw chunks under `_sources`. Keys starting with `_` are for the application; they are not sent to the LLM.

In [ ]:
retriever_tool = RetrieveRelevantChunks(retriever)
print(json.dumps(retriever_tool.json_schema(), indent=2))

result = await retriever_tool.call(
    {"query": "Which pallets in our portfolio have a third-party verified EPD?"}
)
print(result["result"][:2000])

## LLM with the tool

The system prompt says when to use the tool and how to treat its results.

In [ ]:
SYSTEM = """You are an assistant for PrimePack AG.
Call retrieve_relevant_chunks for questions about PrimePack products, suppliers or documents. Answer general questions from your own knowledge.
When you use retrieved chunks:
- Use them as your only source of truth.
- If they do not contain the answer, say that you do not know.
- Cite the source file of every claim.
- Keep the answer concise."""

system_message = text_message(Roles.SYSTEM, SYSTEM)
llm = OpenAILLM(tools=[retriever_tool], tool_choice="auto")

### General question

The LLM should answer directly, without a tool call.

In [ ]:
response = await llm.generate(
    [
        system_message,
        text_message(
            Roles.USER,
            "What is Einstein's theory of relativity? Answer in 2-3 sentences.",
        ),
    ]
)
print(response.content[0].text)
print(f"Tool calls: {response.tool_calls}")

### Domain question

A response with a tool call contains no answer text yet. Compare the `query` argument with the question: the LLM often rephrases it.

In [ ]:
messages = [
    system_message,
    text_message(
        Roles.USER, "Which pallets in our portfolio have a third-party verified EPD?"
    ),
]
response = await llm.generate(messages)

print(f"Content: {response.content[0].text!r}")
for tool_call in response.tool_calls or []:
    print(f"{tool_call.function.name}({tool_call.function.arguments})")

Run each tool call and send its `result` back as a `TOOL` message, as in T10. The final call includes the system prompt, so its rules on grounding and citing apply.

In [ ]:
tools = {tool.name: tool for tool in llm.tools or []}
tool_messages = []
for tool_call in response.tool_calls or []:
    result = await tools[tool_call.function.name].call(
        json.loads(tool_call.function.arguments)
    )
    print(f"{tool_call.id}: {len(result['_sources'])} chunks")
    tool_messages.append(
        LLMMessage(
            role=Roles.TOOL,
            name=tool_call.function.name,
            tool_call_id=tool_call.id,
            content=[MessageContent(type="text", text=result["result"])],
        )
    )

final_response = await llm.generate([*messages, response, *tool_messages])
print(final_response.content[0].text)